# Data preparation: the CAIDA AS-level graph

## Purpose

The solver is validated on the toy graphs of notebook 00; this notebook
turns the real CAIDA snapshot into the directed graph the ranking
notebooks will use. It discharges requirement 1 of the brief at the data
level: model a non-web network as a directed graph, and separate the facts
the data states from the modelling decisions we impose on top.

Every choice is kept as a parameter rather than baked in. Two of them are
open modelling questions (which way a customer-provider link votes, and
whether peer-to-peer links contribute edges), so we build *all* the
variants and tabulate them side by side rather than picking one here. The
ranking notebook reports the alternatives; this notebook only labels one
combination the *provisional primary* so later notebooks have a default to
load.

The computation lives in `pagerank.caida`; this notebook narrates, checks
and plots.

In [ ]:
import hashlib
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pagerank import from_edge_list, top_k
from pagerank.caida import build_edge_list, read_as_rel, save_build

REPO = Path.cwd().parent
RAW = REPO / 'data' / 'raw'
PROCESSED = REPO / 'data' / 'processed'
FIGURES = REPO / 'results' / 'figures'
TABLES = REPO / 'results' / 'tables'

SNAPSHOT = '20260901'
AS_REL = RAW / f'{SNAPSHOT}.as-rel.bz2'
PPDC = RAW / f'{SNAPSHOT}.ppdc-ases.bz2'

## 1. Provenance

Snapshot `20260901`, CAIDA `serial-1`. The raw files are **not** in the
repository: the CAIDA Acceptable Use Agreement governs redistribution, so
`data/raw/` is git-ignored. See `data/README.md` for how to obtain the
snapshot and for the required citation. The size and md5 below pin the
exact bytes this notebook was run against, so a reader who downloads the
same snapshot can confirm it is identical.

In [ ]:
def file_md5(path, chunk=1 << 20):
    digest = hashlib.md5()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(chunk), b''):
            digest.update(block)
    return digest.hexdigest()

for path in (AS_REL, PPDC):
    size_mb = path.stat().st_size / 1e6
    print(f'{path.name:28s} {size_mb:7.3f} MB  md5 {file_md5(path)}')

# The first two header lines, verbatim, as a provenance sanity check.
import bz2
with bz2.open(AS_REL, 'rt', encoding='ascii') as handle:
    for _ in range(2):
        print(handle.readline().rstrip()[:80], '...')

## 2. Reading the file

`read_as_rel` parses the file into a snapshot of raw triples plus the
header, applying no modelling choice. The counts below are asserted equal
to the numbers recorded when the snapshot was first inspected; a mismatch
means the snapshot differs from the one documented and stops the notebook
rather than silently ranking a different graph.

In [ ]:
t0 = time.perf_counter()
snap = read_as_rel(AS_REL)
read_seconds = time.perf_counter() - t0

print(f'read in {read_seconds:.2f} s')
print(f'comment lines      {snap.n_comment_lines}')
print(f'relationships      {snap.relationships.shape[0]}')
print(f'  customer-provider {snap.n_c2p}')
print(f'  peer-to-peer      {snap.n_p2p}')

# Reconcile against the documented snapshot. The snapshot differs from the
# documented one if any of these fail.
assert snap.n_comment_lines == 175, 'comment-line count differs from the documented snapshot'
assert snap.relationships.shape[0] == 525848, 'relationship count differs from the documented snapshot'
assert snap.n_c2p == 166906, 'customer-provider count differs from the documented snapshot'
assert snap.n_p2p == 358942, 'peer-to-peer count differs from the documented snapshot'
assert len(snap.clique) == 18, 'clique size differs from the documented snapshot'
assert len(snap.ixp_asns) == 28, 'IXP list size differs from the documented snapshot'
print('all counts reconcile with the documented snapshot')

## 3. What the header says

Three facts come straight from the header. The **input clique** is the set
of transit-free ASes the inference forces to peer with one another (18
ASNs; see the dataset README). The **IXP ASes** are 28 ASNs that CAIDA
flags as internet-exchange route servers rather than ordinary networks,
which is why they are a candidate for exclusion later. The **source** lines
record which BGP collectors and dates fed the inference.

In [ ]:
print(f'input clique ({len(snap.clique)} ASNs):')
print(' ', ' '.join(str(a) for a in snap.clique))
print(f'\nIXP ASes ({len(snap.ixp_asns)} ASNs):')
print(' ', ' '.join(str(a) for a in snap.ixp_asns))

In [ ]:
# Each source line is '|'-separated: topology | BGP | date | project | collector.
source_rows = [s.split('|') for s in snap.sources]
sources = pd.DataFrame(source_rows)
sources.columns = [f'f{i}' for i in range(sources.shape[1])]
# Summarise as collector x number of dated observations.
summary = (
    sources.groupby([sources.columns[-2], sources.columns[-1]])
    .size()
    .rename('n_source_lines')
    .reset_index()
)
print(f'{len(snap.sources)} source lines')
summary

## 4. Step tags

Since December 2022 CAIDA tags each relationship with the inference step
that produced it, 1 to 8. Lower-numbered steps rest on stronger evidence
(step 1 is the forced clique; step 8 is the catch-all peering fallback), so
the tag is a per-link confidence signal. We keep it for the write-up but do
not weight edges by it: the brief's PageRank is unweighted.

In [ ]:
step_df = pd.DataFrame({'step': snap.step, 'rel': snap.relationships[:, 2]})
step_df['kind'] = np.where(step_df['rel'] == -1, 'c2p', 'p2p')
by_step = (
    step_df.groupby(['step', 'kind']).size().unstack(fill_value=0)
)
by_step['total'] = by_step.sum(axis=1)
by_step

## 5. ASNs and indexing

The nodes are the distinct ASNs appearing in the relationships. ASNs are
not contiguous: the maximum ASN is far larger than the node count, so a
dense array indexed by ASN over `0..max_asn` would be mostly empty. We
therefore index by position in the sorted ASN array and look ASNs up with a
binary search (`np.searchsorted`), paid once per edge at build time. The
ratio below is how much a dense array would waste.

In [ ]:
all_asns = np.unique(snap.relationships[:, :2])
n_distinct = all_asns.shape[0]
max_asn = int(all_asns.max())
print(f'distinct ASNs : {n_distinct}')
print(f'maximum ASN   : {max_asn}')
print(f'density       : {n_distinct / (max_asn + 1):.4f} '
      f'(a dense 0..max_asn array would be {100 * (1 - n_distinct / (max_asn + 1)):.1f}% empty)')

## 6. Four graph variants

Two modelling questions are still open, so we build both options of each
and report them side by side (vote fixed to `customer_to_provider`, the
reading argued for in the lab notebook: a customer paying for transit
endorses its provider).

- **peering** `both` vs `none`: whether a peer-to-peer link adds two edges
  or none.
- **IXPs** kept vs excluded: whether the 28 IXP route-server ASes stay in
  the graph.

Each variant is passed through `from_edge_list`, which removes self-loops
and exact duplicate edges and counts them, and reports the dangling nodes
(no out-link), which is what the solver's dangling treatment exists for.

In [ ]:
VOTE = 'customer_to_provider'
ixp = list(snap.ixp_asns)

variants = {
    ('both', 'kept'): dict(peering='both', exclude_asns=()),
    ('both', 'excluded'): dict(peering='both', exclude_asns=ixp),
    ('none', 'kept'): dict(peering='none', exclude_asns=()),
    ('none', 'excluded'): dict(peering='none', exclude_asns=ixp),
}

builds = {}
graphs = {}
rows = []
for (peering, ixp_label), kwargs in variants.items():
    build = build_edge_list(snap, vote=VOTE, **kwargs)
    graph = from_edge_list(
        [(int(t), int(h)) for t, h in build.edges], build.n
    )
    builds[(peering, ixp_label)] = build
    graphs[(peering, ixp_label)] = graph
    n_dangling = int(graph.dangling.sum())
    rows.append({
        'peering': peering,
        'ixps': ixp_label,
        'nodes': graph.n,
        'edges': graph.n_edges,
        'self_loops_dropped': graph.n_self_loops_dropped,
        'duplicates_dropped': graph.n_duplicates_dropped,
        'dangling': n_dangling,
        'dangling_share': n_dangling / graph.n,
        'asns_dropped': build.n_asns_dropped,
    })

table = pd.DataFrame(rows)
table

In [ ]:
TABLES.mkdir(parents=True, exist_ok=True)
csv_path = TABLES / '01_graph_variants.csv'
table.to_csv(csv_path, index=False)
print('wrote', csv_path.relative_to(REPO))

In [ ]:
# A fact from the data worth stating outright: do the 28 IXP ASes from the
# header actually appear in any relationship line?
all_rel_asns = np.unique(snap.relationships[:, :2])
ixp_present = np.isin(np.array(snap.ixp_asns), all_rel_asns)
print(f'IXP ASes from the header that appear in a relationship: '
      f'{int(ixp_present.sum())} of {len(snap.ixp_asns)}')

Three things to read off the table.

Under **peering `none`** the dangling count jumps from 0 to 494: with no
peering edges, every AS whose only links were peering links loses its sole
out-direction and leaks mass. (The clique itself is not among them here --
clique members still have customer-provider edges pointing away from them
-- but the broader set of peering-only ASes does dangle.) That leak is a
strong argument against `none` as the primary model.

**Excluding IXPs makes no difference in this snapshot**, and that is a fact
about the data rather than a modelling choice. The cell above shows that
none of the 28 header-declared IXP ASes appear in any relationship line:
CAIDA flags them in the header but infers no business relationship for
them, so there is nothing to drop. The `kept` and `excluded` rows are
therefore identical and `asns_dropped` is 0. We still build and save both
variants so the ranking notebook can state the exclusion was examined and
found inert, rather than silently skipping it.

We label **peering `both`, IXPs kept** the *provisional primary* variant,
explicitly pending the modelling decision that is not this notebook's to
make. The ranking notebook will report the alternatives against it.

In [ ]:
PRIMARY = ('both', 'kept')
primary_build = builds[PRIMARY]
primary_graph = graphs[PRIMARY]
print('provisional primary:', PRIMARY)
print(f'  nodes {primary_graph.n}, edges {primary_graph.n_edges}, '
      f'dangling {int(primary_graph.dangling.sum())}')

## 7. Degree distributions

For the provisional primary variant, the complementary cumulative
distribution (CCDF) of in-degree and out-degree on log-log axes. The AS
graph is known to be heavy-tailed: a few ASes have enormous degree and most
have very little, which is the structural reason PageRank and in-degree
will mostly agree at the top and disagree in the tail (requirement 4). No
interpretation beyond noting whether clique members and IXPs show up among
the highest-degree nodes; that is the ranking notebook's job.

In [ ]:
def ccdf(values):
    values = np.sort(values)
    n = values.size
    # P(X >= x) at each distinct value.
    unique, idx = np.unique(values, return_index=True)
    survival = 1.0 - idx / n
    return unique, survival

in_deg = primary_graph.in_degree
out_deg = primary_graph.out_degree

fig, ax = plt.subplots(figsize=(6, 4))
for deg, label in ((in_deg, 'in-degree'), (out_deg, 'out-degree')):
    x, y = ccdf(deg[deg > 0])
    ax.loglog(x, y, marker='.', linestyle='none', markersize=3, label=label)
ax.set_xlabel('degree k')
ax.set_ylabel('P(degree >= k)')
ax.set_title('Degree CCDF, provisional primary variant (peering both, IXPs kept)')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
fig.tight_layout()

FIGURES.mkdir(parents=True, exist_ok=True)
png_path = FIGURES / '01_degree_distribution.png'
fig.savefig(png_path, dpi=150)
plt.show()
print('wrote', png_path.relative_to(REPO), f'({png_path.stat().st_size / 1024:.0f} KB)')

In [ ]:
asns = primary_build.asns
clique_set = set(snap.clique)
ixp_set = set(snap.ixp_asns)

def annotate(asn):
    tags = []
    if asn in clique_set:
        tags.append('clique')
    if asn in ixp_set:
        tags.append('IXP')
    return ','.join(tags) if tags else ''

print('Top 10 by in-degree:')
for idx, deg in top_k(in_deg.astype(float), 10):
    asn = int(asns[idx])
    print(f'  AS{asn:<8d} in-degree {int(deg):6d}  {annotate(asn)}')

print('\nTop 10 by out-degree:')
for idx, deg in top_k(out_deg.astype(float), 10):
    asn = int(asns[idx])
    print(f'  AS{asn:<8d} out-degree {int(deg):6d}  {annotate(asn)}')

## 8. Saving the four variants

Each build is written to `data/processed/` under a name that records every
modelling choice, so a later notebook loads exactly the graph it means to.
Nothing under `data/` is tracked by git (AUA plus size); these artefacts
are rebuilt by re-running this notebook.

In [ ]:
PROCESSED.mkdir(parents=True, exist_ok=True)

def variant_filename(peering, ixp_label):
    ixp_tag = 'kept' if ixp_label == 'kept' else 'excluded'
    return f'{SNAPSHOT}_peering-{peering}_vote-c2p_ixp-{ixp_tag}.npz'

for (peering, ixp_label), build in builds.items():
    name = variant_filename(peering, ixp_label)
    save_build(build, PROCESSED / name)
    size_kb = (PROCESSED / name).stat().st_size / 1024
    print(f'{name:52s} {size_kb:8.1f} KB')

## 9. What this notebook established

- Snapshot `20260901`, CAIDA `serial-1`, reconciled to the documented
  counts: 175 comment lines, 525,848 relationships (166,906 customer-
  provider, 358,942 peer-to-peer), an 18-ASN input clique and 28 IXP ASes.
- Nodes are the distinct ASNs, indexed by position in the sorted ASN array;
  a dense array over the ASN space is rejected as mostly empty.
- Four graph variants were built and saved (peering both/none x IXPs
  kept/excluded, vote customer-to-provider). Their node, edge and dangling
  counts are in `results/tables/01_graph_variants.csv`.
- Peering `none` raises the dangling count from 0 to 494 (peering-only ASes
  lose their sole out-direction), the structural reason it is not the
  provisional primary.
- Excluding the 28 header-declared IXP ASes is inert for this snapshot:
  none of them appear in any relationship, so the kept and excluded
  variants are identical. This is a fact about the data, recorded rather
  than assumed. `both` with IXPs kept is the provisional primary, pending
  the modelling decision.
- The degree distribution is heavy-tailed in both directions
  (`results/figures/01_degree_distribution.png`), setting up the PageRank
  versus in-degree comparison in the ranking notebook.

The modelling choices are deliberately not resolved here. That is the
ranking notebook's call, informed by these counts.